In [1]:
import pandas as pd
df=pd.read_csv('../data/processed/ipl_cleaned.csv')
df

C:\Users\dasra\AppData\Local\Temp\ipykernel_14024\17270735.py:2: DtypeWarning: Columns (27,28,42,45,46,47,50) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv('../data/processed/ipl_cleaned.csv')


,match_id,date,match_type,event_name,innings,batting_team,bowling_team,over,ball,ball_no,...,batter_runs,batter_balls,bowler_wicket,batting_partners,next_batter,striker_out,total_balls,is_boundary,is_wicket,strike_rate
0,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,0.1,...,0,1,0,"('BB McCullum', 'SC Ganguly')",NaN,False,1,0,0,0.0
1,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,0.2,...,0,1,0,"('BB McCullum', 'SC Ganguly')",NaN,False,2,0,0,0.0
2,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,0.3,...,0,1,0,"('BB McCullum', 'SC Ganguly')",NaN,False,3,0,0,NaN
3,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,0.3,...,0,2,0,"('BB McCullum', 'SC Ganguly')",NaN,False,3,0,0,0.0
4,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,0.4,...,0,3,0,"('BB McCullum', 'SC Ganguly')",NaN,False,4,0,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
283673,1529267,2026-04-16,T20,Indian Premier League,2,Punjab Kings,Mumbai Indians,16,1,16.1,...,2,2,0,"('MP Stoinis', 'P Simran Singh')",NaN,False,97,0,0,NaN
283674,1529267,2026-04-16,T20,Indian Premier League,2,Punjab Kings,Mumbai Indians,16,1,16.1,...,2,3,0,"('MP Stoinis', 'P Simran Singh')",NaN,False,97,0,0,0.0
283675,1529267,2026-04-16,T20,Indian Premier League,2,Punjab Kings,Mumbai Indians,16,2,16.2,...,6,4,0,"('MP Stoinis', 'P Simran Singh')",NaN,False,98,1,0,400.0
283676,1529267,2026-04-16,T20,Indian Premier League,2,Punjab Kings,Mumbai Indians,16,3,16.3,...,6,4,0,"('MP Stoinis', 'P Simran Singh')",NaN,False,99,0,0,NaN


In [2]:
match_df = df[['match_id','batting_team','bowling_team','venue',
               'toss_winner','toss_decision','match_won_by']].drop_duplicates()

In [3]:
match_df['toss_win_match_win'] = (match_df['toss_winner'] == match_df['match_won_by']).astype(int)

In [4]:
venue_win = match_df.groupby(['venue','match_won_by']).size().reset_index(name='wins')

total_venue = match_df.groupby('venue').size().reset_index(name='total')

venue_win = venue_win.merge(total_venue, on='venue')
venue_win['venue_win_pct'] = venue_win['wins'] / venue_win['total']

In [5]:
team_wins = match_df['match_won_by'].value_counts()

team_matches = pd.concat([match_df['batting_team'], match_df['bowling_team']]).value_counts()

team_win_pct = (team_wins / team_matches).fillna(0)

In [6]:
match_df['team1_win_pct'] = match_df['batting_team'].map(team_win_pct)
match_df['team2_win_pct'] = match_df['bowling_team'].map(team_win_pct)

In [7]:
h2h = match_df.groupby(['batting_team','bowling_team','match_won_by']).size().reset_index(name='wins')

In [8]:
match_df['target'] = (match_df['batting_team'] == match_df['match_won_by']).astype(int)

In [9]:
final_df = match_df[['batting_team','bowling_team','venue',
                     'toss_winner','toss_decision',
                     'team1_win_pct','team2_win_pct',
                     'toss_win_match_win','target']]

In [10]:
final_df.to_csv('../data/processed/ipl_features.csv', index=False)